# MLflow Logging for Final XGBoost Model

This notebook trains the final tuned XGBoost Classifier and logs all parameters, metrics, models, and artifacts (e.g., ROC curve, Confusion Matrix) to MLflow. It also incorporates threshold optimization.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score, 
    accuracy_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve
)

import mlflow
import mlflow.xgboost
import mlflow.sklearn
from mlflow.models.signature import infer_signature

## Set Up MLflow

In [2]:
# Connect to the local SQLite database used for tracking
mlflow.set_tracking_uri("sqlite:///mlflow.db")

# Define the experiment name
experiment_name = "Home Credit Default Risk"
mlflow.set_experiment(experiment_name)

<Experiment: artifact_location='file:d:/MLPs/Home Credit Defaut Risk Predictor/mlruns/1', creation_time=1781026449802, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1781026449802, lifecycle_stage='active', name='Home Credit Default Risk', tags={}, trace_location=None, workspace='default'>

## Load and Split Data

In [3]:
print("Loading data...")
train = pd.read_csv('train_fe.csv')

X = train.drop(columns=['SK_ID_CURR', 'TARGET'])
y = train['TARGET']

RANDOM_STATE = 42
TEST_SIZE = 0.2

print("Splitting data...")
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Validation set shape: {X_val.shape}")

Loading data...
Splitting data...
Training set shape: (246008, 64)
Validation set shape: (61503, 64)


## Train Model, Optimize Threshold, and Log to MLflow

In [4]:
# Best parameters found during hyperparameter tuning
best_params = {
    'colsample_bytree': 0.8853,
    'gamma': 1.5216,
    'learning_rate': 0.0675,
    'max_depth': 5,
    'min_child_weight': 1,
    'n_estimators': 114,
    'scale_pos_weight': 6.1089,
    'subsample': 0.7757,
    'random_state': RANDOM_STATE,
    'eval_metric': 'auc',
}

with mlflow.start_run(run_name="Final_Tuned_XGBoost"):
    # 1. Log Parameters
    mlflow.log_params(best_params)
    
    # 2. Train the Model
    print("Training XGBoost model...")
    model = XGBClassifier(**best_params)
    model.fit(X_train, y_train)
    
    # 3. Predict Probabilities
    print("Evaluating model...")
    y_pred_proba = model.predict_proba(X_val)[:, 1]
    
    # Calculate Default Metrics (Threshold = 0.5)
    y_pred_default = model.predict(X_val)
    roc_auc = roc_auc_score(y_val, y_pred_proba)
    
    # Threshold Optimization using Youden's J statistic
    fpr, tpr, thresholds = roc_curve(y_val, y_pred_proba)
    J = tpr - fpr
    best_thresh_idx = np.argmax(J)
    best_thresh = thresholds[best_thresh_idx]
    
    print(f"Optimized Threshold (Youden's J): {best_thresh:.4f}")
    
    # Predict using the optimized threshold
    y_pred_opt = (y_pred_proba >= best_thresh).astype(int)
    
    # Calculate Metrics at Optimized Threshold
    opt_accuracy = accuracy_score(y_val, y_pred_opt)
    opt_precision = precision_score(y_val, y_pred_opt)
    opt_recall = recall_score(y_val, y_pred_opt)
    opt_f1 = f1_score(y_val, y_pred_opt)
    
    # 4. Log Metrics
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("best_threshold", best_thresh)
    mlflow.log_metric("opt_accuracy", opt_accuracy)
    mlflow.log_metric("opt_precision", opt_precision)
    mlflow.log_metric("opt_recall", opt_recall)
    mlflow.log_metric("opt_f1_score", opt_f1)
    
    print(f"Validation ROC-AUC: {roc_auc:.4f}")
    print(f"Optimized Accuracy: {opt_accuracy:.4f}")
    print(f"Optimized F1 Score: {opt_f1:.4f}")
    
    # 5. Log Model with Signature
    signature = infer_signature(X_val, y_pred_opt)
    mlflow.xgboost.log_model(model.get_booster(), "xgboost-model", signature=signature)
    mlflow.sklearn.log_model(model, "sklearn-xgboost-model", signature=signature)
    
    # 6. Generate and Log Plots
    # Confusion Matrix (Optimized)
    plt.figure(figsize=(6, 5))
    cm = confusion_matrix(y_val, y_pred_opt)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix (Threshold = {best_thresh:.4f})')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig("confusion_matrix_opt.png")
    mlflow.log_artifact("confusion_matrix_opt.png")
    plt.close()
    
    # ROC Curve
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f'XGBoost (AUC = {roc_auc:.4f})')
    plt.scatter(fpr[best_thresh_idx], tpr[best_thresh_idx], marker='o', color='red', label='Best Threshold')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig("roc_curve.png")
    mlflow.log_artifact("roc_curve.png")
    plt.close()
    
    print("Run logged successfully to MLflow!")

2026/06/09 23:19:34 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



Training XGBoost model...
Evaluating model...
Optimized Threshold (Youden's J): 0.3689
Validation ROC-AUC: 0.7633
Optimized Accuracy: 0.7231
Optimized F1 Score: 0.2794


2026/06/09 23:19:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/09 23:19:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/09 23:19:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run logged successfully to MLflow!
